In [ ]:
import pandas as pd
import pytz, datetime, uuid
import sys
import os
from pathlib import Path
from time import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Aseguramos que el Notebook vea la carpeta actual
sys.path.append(os.getcwd())

# 2. IMPORTAMOS LAS FUNCIONES (Aquí estaba el fallo)
try:
    from utils_sheet import first_session, left_to_rigth, right_to_left
    print("✅ Funciones de 'utils.py' cargadas correctamente.")
except ImportError as e:
    print(f"❌ Error al importar utils: {e}")
    # Si falla por el punto en utils.py, asegúrate de haber quitado 
    # el '.' en la línea 9 de utils.py como vimos antes.

# --- CONFIGURACIÓN ---
DATA_PATH = Path("./data_scraping.csv")
GMT5 = pytz.timezone("Etc/GMT+5")
today = datetime.datetime.now(GMT5).strftime("%d-%m-%Y")

DEPS = [str(n).zfill(2) for n in range(1, 26) if n != 15]
LIMA = "15"

def ejecutar(dep):
    session = None
    try:
        # Ahora sí, first_session debería estar definido
        session, _, view_state = first_session()
        data = left_to_rigth(view_state, dep, session)
        return data
    except Exception as e:
        print(f"❌ Error en dep {dep}: {e}")
        return pd.DataFrame()
    finally:
        if session and hasattr(session, 'quit'):
            session.quit()

def both_sides(right=True):
    session = None
    try:
        session, _, view_state = first_session()
        if right:
            return left_to_rigth(view_state, LIMA, session, right=right)
        else:
            return right_to_left(view_state, LIMA, session)
    except Exception as e:
        print(f"❌ Error en Lima (right={right}): {e}")
        return pd.DataFrame()
    finally:
        if session and hasattr(session, 'quit'):
            session.quit()

# --- EJECUCIÓN ---
if __name__ == "__main__":
    start = time()
    data_list = []

    print(f"🚀 Iniciando scraping en paralelo (2 workers para estabilidad)...")

    # Usamos 2 workers para evitar que Chrome se bloquee en el Notebook
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(ejecutar, dep): dep for dep in DEPS}
        for future in as_completed(futures):
            res = future.result()
            if res is not None and not res.empty:
                data_list.append(res)

    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(both_sides, side): side for side in [True, False]}
        for future in as_completed(futures):
            res = future.result()
            if res is not None and not res.empty:
                data_list.append(res)

    # --- GUARDAR RESULTADOS ---
    if data_list:
        data_final = pd.concat(data_list, ignore_index=True)
        data_final["id_uuid"] = [uuid.uuid4() for _ in range(len(data_final))]
        data_final["fechaActualizacion"] = datetime.datetime.now()
        
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        data_final.to_csv(DATA_PATH, index=False)
        print(f"✅ DATOS GUARDADOS: {len(data_final)} registros.")
    else:
        print("⚠️ No se obtuvieron datos. Revisa si hubo errores de conexión arriba.")

    print(f"⏱️ Tiempo total: {time() - start:.2f} segundos")


✅ Funciones de 'utils.py' cargadas correctamente.
🚀 Iniciando scraping en paralelo (2 workers para estabilidad)...


Scraping Inverso Dep 15: 100%|██████████| 90/90 [01:42<00:00,  1.14s/it]

✅ DATOS GUARDADOS: 5290 registros.
⏱️ Tiempo total: 345.68 segundos


In [4]:
len(data_final)

5193

In [2]:
data_final[data_final['posicion']== 'EJECUTOR COACTIVO']


,posicion,empresa,ubicacion,num_conv,n_vac,salario,fecha_inicio,fecha_fin,id_uuid,fechaActualizacion
659,EJECUTOR COACTIVO,SERVICIO DE ADMINISTRACION TRIBUTARIA- CAJAMARCA,CAJAMARCA - CAJAMARCA,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,2,"S/. 3,500.00",12/03/2026,25/03/2026,5469dfe6-a316-4b2c-9c09-a0dcc3c0d487,2026-03-22 15:11:48.670493
1267,EJECUTOR COACTIVO,MUNICIPALIDAD DISTRITAL DE MOLINO,HUANUCO - MOLINO,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 2,100.00",12/03/2026,25/03/2026,f803b060-82f0-4703-8034-f4fa3a63e573,2026-03-22 15:11:48.670493
2192,EJECUTOR COACTIVO,MUNICIPALIDAD PROVINCIAL SANCHEZ CARRION,LA LIBERTAD - HUAMACHUCO,D.LEG 1057 - DETERMINADO (SUPLENCIA)-001,1,"S/. 2,735.48",12/03/2026,25/03/2026,e0d745fd-fba9-4704-a0f9-4bcde24fb16e,2026-03-22 15:11:48.670493
2527,EJECUTOR COACTIVO,MUNICIPALIDAD PROVINCIAL JAUJA,JUNIN - JAUJA,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 3,500.00",12/03/2026,25/03/2026,6d465367-9f75-42f5-b4a3-570c7ba5140e,2026-03-22 15:11:48.670493
2921,EJECUTOR COACTIVO,MUNICIPALIDAD DIST. DE LOS ORGANOS,PIURA - LOS ORGANOS,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 4,500.00",20/03/2026,06/04/2026,78af31c7-8577-4e9d-b9d2-800db719c40f,2026-03-22 15:11:48.670493
2960,EJECUTOR COACTIVO,PROCURADORIA DE LA MUNICIPALIDAD DISTRITAL VEI...,PIURA - VEINTISEIS DE OCTUBRE,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 3,000.00",11/03/2026,24/03/2026,4a88a1a5-e528-45ba-a8df-83a0779b845c,2026-03-22 15:11:48.670493
3750,EJECUTOR COACTIVO,MUNICIPALIDAD DE CHORRILLOS,LIMA - CHORRILLOS,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 5,000.00",16/03/2026,27/03/2026,b1a72607-af23-41ca-af94-fa15e41b17cc,2026-03-22 15:11:48.670493
4012,EJECUTOR COACTIVO,MINISTERIO DE TRANSPORTES Y COMUNICACIONES,LIMA - LIMA,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 11,864.19",11/03/2026,24/03/2026,fb5757b4-8d71-462b-bba9-23df6c602147,2026-03-22 15:11:48.670493
4506,EJECUTOR COACTIVO,MINISTERIO DE TRANSPORTES Y COMUNICACIONES,LIMA - LIMA,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 11,864.19",11/03/2026,24/03/2026,2c1ad601-b88b-4574-9b27-056771120be5,2026-03-22 15:11:48.670493
4764,EJECUTOR COACTIVO,MUNICIPALIDAD DE CHORRILLOS,LIMA - CHORRILLOS,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 5,000.00",16/03/2026,27/03/2026,5516fcb0-b8a7-4331-a141-db52d49580a8,2026-03-22 15:11:48.670493


In [3]:
# filtra df_final con salario = S/. 2,864.19
data_final[data_final['posicion'].str.contains('OMAPED', case=False, na=False)]

,posicion,empresa,ubicacion,num_conv,n_vac,salario,fecha_inicio,fecha_fin,id_uuid,fechaActualizacion
395,RESPONSABLE DE CIAM Y OMAPED,MUNICIPALIDAD DISTRITAL DE VINCHOS,AYACUCHO - VINCHOS,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 2,500.00",17/03/2026,30/03/2026,aee0d66b-1791-4331-86f0-44d27086272a,2026-03-22 15:11:48.670493
455,RESPONSABLE DE CIAM - OMAPED,MUNICIPALIDAD DISTRITAL DE LURICOCHA,AYACUCHO - LURICOCHA,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 1,400.00",16/03/2026,27/03/2026,19f3c320-016c-4aa0-b5db-d94128c01701,2026-03-22 15:11:48.670493
1137,RESPONSABLE DE OMAPED Y CIAM,MUNICIPALIDAD DISTRITAL DE CHAGLLA,HUANUCO - CHAGLLA,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 2,000.00",17/03/2026,30/03/2026,72d4e009-ebe3-4260-b3c6-a6a1f08e4683,2026-03-22 15:11:48.670493
1161,COORDINADOR DE LA UNIDAD DE OMAPED,MUNICIPALIDAD PROVINCIAL DE PACHITEA,HUANUCO - PANAO,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 2,100.00",17/03/2026,30/03/2026,db30e6d1-d30e-41d0-9e7d-e4f176647322,2026-03-22 15:11:48.670493
1224,RESPONSABLE DE OMAPED,MUNICIPALIDAD PROVINCIAL DEL MARANON,HUANUCO - HUACRACHUCO,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 1,600.00",16/03/2026,27/03/2026,85b3d438-dd6f-4aa8-bb96-64d858f43ca9,2026-03-22 15:11:48.670493
1639,RESPONSABLE DE LA UNIDAD FUNCIONAL DEMUNA Y OM...,MUNICIPALIDAD DISTRITAL DE POMACANCHI,CUSCO - POMACANCHI,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 3,000.00",16/03/2026,27/03/2026,40e22123-0310-45b0-8983-6a8361fbf0b4,2026-03-22 15:11:48.670493
1657,JEFE DE LA OFICINA DE DEMUNA Y OMAPED,MUNICIPALIDAD DISTRITAL DE HUANOQUITE,CUSCO - HUANOQUITE,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 2,650.00",13/03/2026,26/03/2026,ede84aaf-f626-4718-82e6-7f46997fdbcb,2026-03-22 15:11:48.670493
1901,RESPONSABLE DE OMAPED-PROFESIONAL II,MUNICIPALIDAD DISTRITAL SAN SEBASTIAN,CUSCO - SAN SEBASTIAN,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 3,500.00",11/03/2026,24/03/2026,0590f828-6e4f-4cdc-b704-5e4c8f7cb208,2026-03-22 15:11:48.670493
2012,PERSONAL DE APOYO ADMINISTRATIVO - OMAPED,MUNICIPALIDAD DISTRITAL DE SAN SALVADOR,CUSCO - SAN SALVADOR,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 1,800.00",11/03/2026,24/03/2026,23d3c43e-47f9-455c-90d9-66dd3587d097,2026-03-22 15:11:48.670493
2155,"JEFE DE LA DIVISION DE DEMUNA, OMAPED Y CIAM",MUNICIPALIDAD DISTRITAL MARCABAL,LA LIBERTAD - MARCABAL,D.LEG 1057 - DETERMINADO (NECESIDAD TRANSITORI...,1,"S/. 2,800.19",19/03/2026,01/04/2026,7275213e-cab5-4c0d-b614-aa27c66f04c9,2026-03-22 15:11:48.670493


In [ ]:
import os

# Ruta al archivo (basado en tu error anterior)
utils_path = 'utils.py'

if os.path.exists(utils_path):
    with open(utils_path, 'r') as f:
        contenido = f.read()
    
    # Cambiamos el import relativo por uno absoluto
    if 'from .config' in contenido:
        nuevo_contenido = contenido.replace('from .config', 'from config')
        with open(utils_path, 'w') as f:
            f.write(nuevo_contenido)
        print("✅ 'utils.py' corregido: se eliminó el import relativo.")
    else:
        print("ℹ️ El archivo ya no tiene imports relativos o ya fue corregido.")
else:
    print("❌ No se encontró 'utils.py' en esta carpeta.")